# AEI Report v3 Claude.ai Preprocessing

This notebook takes processed Clio data and enriches it with external sources:
1. Merges with population data for per capita calculations
2. Merges with GDP data for economic analysis
3. Merges with SOC/O*NET data for occupational analysis
4. Applies MIN_OBSERVATIONS filtering
5. Calculates derived metrics (per capita, indices, tiers)
6. Categorizes collaboration patterns

**Input**: `aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv`

**Output**: `aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv`

## Configuration and Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Year for external data
YEAR = 2024

# Data paths - using local directories
DATA_INPUT_DIR = "../data/input"  # Raw external data
DATA_INTERMEDIATE_DIR = (
    "../data/intermediate"  # Processed external data and Clio output
)
DATA_OUTPUT_DIR = "../data/output"  # Final enriched data

# Minimum observation thresholds
MIN_OBSERVATIONS_COUNTRY = 200  # Threshold for countries
MIN_OBSERVATIONS_US_STATE = 100  # Threshold for US states

In [ ]:
# Countries where Claude doesn't operate (23 countries)
EXCLUDED_COUNTRIES = [
    "AF",  # Afghanistan
    "BY",  # Belarus
    "CD",  # Democratic Republic of the Congo
    "CF",  # Central African Republic
    "CN",  # China
    "CU",  # Cuba
    "ER",  # Eritrea
    "ET",  # Ethiopia
    "HK",  # Hong Kong
    "IR",  # Iran
    "KP",  # North Korea
    "LY",  # Libya
    "ML",  # Mali
    "MM",  # Myanmar
    "MO",  # Macau
    "NI",  # Nicaragua
    "RU",  # Russia
    "SD",  # Sudan
    "SO",  # Somalia
    "SS",  # South Sudan
    "SY",  # Syria
    "VE",  # Venezuela
    "YE",  # Yemen
]

## Data Loading Functions

In [ ]:
def load_population_data():
    """
    Load population data for countries and US states.

    Args:
        verbose: Whether to print progress

    Returns:
        Dict with country and state_us population dataframes
    """
    pop_country_path = (
        Path(DATA_INTERMEDIATE_DIR) / f"working_age_pop_{YEAR}_country.csv"
    )
    pop_state_path = (
        Path(DATA_INTERMEDIATE_DIR) / f"working_age_pop_{YEAR}_us_state.csv"
    )

    if not pop_country_path.exists() or not pop_state_path.exists():
        raise FileNotFoundError(
            f"Population data is required but not found.\n"
            f"  Expected files:\n"
            f"    - {pop_country_path}\n"
            f"    - {pop_state_path}\n"
            f"  Run preprocess_population.py first to generate these files."
        )

    # Use keep_default_na=False to preserve any "NA" values as strings
    df_pop_country = pd.read_csv(
        pop_country_path, keep_default_na=False, na_values=[""]
    )
    df_pop_state = pd.read_csv(pop_state_path, keep_default_na=False, na_values=[""])

    return {"country": df_pop_country, "state_us": df_pop_state}


def load_gdp_data():
    """
    Load GDP data for countries and US states.

    Returns:
        Dict with country and state_us GDP dataframes
    """
    gdp_country_path = Path(DATA_INTERMEDIATE_DIR) / f"gdp_{YEAR}_country.csv"
    gdp_state_path = Path(DATA_INTERMEDIATE_DIR) / f"gdp_{YEAR}_us_state.csv"

    if not gdp_country_path.exists() or not gdp_state_path.exists():
        raise FileNotFoundError(
            f"GDP data is required but not found.\n"
            f"  Expected files:\n"
            f"    - {gdp_country_path}\n"
            f"    - {gdp_state_path}\n"
            f"  Run preprocess_gdp.py first to generate these files."
        )

    # Use keep_default_na=False to preserve any "NA" values as strings
    df_gdp_country = pd.read_csv(
        gdp_country_path, keep_default_na=False, na_values=[""]
    )
    df_gdp_state = pd.read_csv(gdp_state_path, keep_default_na=False, na_values=[""])

    return {"country": df_gdp_country, "state_us": df_gdp_state}


def load_task_data():
    """
    Load O*NET task statements with SOC codes.

    Returns:
        DataFrame with O*NET tasks and SOC major groups
    """
    onet_path = Path(DATA_INTERMEDIATE_DIR) / "onet_task_statements.csv"

    if not onet_path.exists():
        raise FileNotFoundError(
            f"O*NET data is required but not found.\n"
            f"  Expected file:\n"
            f"    - {onet_path}\n"
            f"  Run preprocess_onet.py first to generate this file."
        )

    # Use keep_default_na=False to preserve any "NA" values as strings
    df_onet = pd.read_csv(onet_path, keep_default_na=False, na_values=[""])

    # Normalize task names for matching with Clio data
    df_onet["task_normalized"] = df_onet["Task"].str.lower().str.strip()

    return df_onet


def load_soc_data():
    """
    Load SOC structure data for occupation names.

    Returns:
        DataFrame with SOC major groups and their titles
    """
    soc_path = Path(DATA_INTERMEDIATE_DIR) / "soc_structure.csv"

    if not soc_path.exists():
        raise FileNotFoundError(
            f"SOC structure data is required but not found.\n"
            f"  Expected file:\n"
            f"    - {soc_path}\n"
            f"  Run preprocess_onet.py first to generate this file."
        )

    # Use keep_default_na=False to preserve any "NA" values as strings
    df_soc = pd.read_csv(soc_path, keep_default_na=False, na_values=[""])

    # Get unique major groups with their titles for SOC name mapping
    df_major_groups = df_soc[df_soc["soc_major_group"].notna()][
        ["soc_major_group", "SOC or O*NET-SOC 2019 Title"]
    ].drop_duplicates(subset=["soc_major_group"])

    return df_major_groups


def load_external_data():
    """
    Load all external data sources from local files.

    Returns:
        Dict with population, gdp, task_statements, and soc_structure dataframes
    """

    external_data = {}

    # Load each data source with its specific function
    external_data["population"] = load_population_data()
    external_data["gdp"] = load_gdp_data()
    external_data["task_statements"] = load_task_data()
    external_data["soc_structure"] = load_soc_data()

    return external_data

## Filtering Functions

In [ ]:
def get_filtered_geographies(df):
    """
    Get lists of countries and states that meet MIN_OBSERVATIONS thresholds.

    This function does NOT filter the dataframe - it only identifies which
    geographies meet the thresholds. The full dataframe is preserved
    so we can still report statistics for all geographies.

    Args:
        df: Input dataframe

    Returns:
        Tuple of (filtered_countries list, filtered_states list)
    """
    # Get country usage counts
    country_usage = df[
        (df["facet"] == "country") & (df["variable"] == "usage_count")
    ].set_index("geo_id")["value"]

    # Get state usage counts
    state_usage = df[
        (df["facet"] == "state_us") & (df["variable"] == "usage_count")
    ].set_index("geo_id")["value"]

    # Get countries that meet MIN_OBSERVATIONS threshold
    filtered_countries = country_usage[
        country_usage >= MIN_OBSERVATIONS_COUNTRY
    ].index.tolist()

    # Get states that meet MIN_OBSERVATIONS threshold
    filtered_states = state_usage[
        state_usage >= MIN_OBSERVATIONS_US_STATE
    ].index.tolist()

    return filtered_countries, filtered_states

## Data Merge Functions

In [ ]:
def merge_population_data(df, population_data):
    """
    Merge population data in long format.

    This function:
    1. Adds countries/states that have population but no usage (with 0 usage values)
    2. Adds population as new rows with variable="working_age_pop"

    Args:
        df: Input dataframe in long format
        population_data: Dict with country and state_us population dataframes

    Returns:
        Dataframe with all geographies and population added as rows
    """
    df_result = df.copy()
    new_rows = []

    # Get unique date/platform combinations to replicate for new data
    date_platform_combos = df_result[
        ["date_start", "date_end", "platform_and_product"]
    ].drop_duplicates()

    # Process countries
    if "country" in population_data and not population_data["country"].empty:
        pop_country = population_data["country"]

        # Get existing countries in our data
        existing_countries = df_result[
            (df_result["geography"] == "country")
            & (df_result["variable"] == "usage_count")
        ]["geo_id"].unique()

        # Add missing countries with 0 usage (excluding excluded countries)
        missing_countries = (
            set(pop_country["country_code"])
            - set(existing_countries)
            - set(EXCLUDED_COUNTRIES)
        )

        for _, combo in date_platform_combos.iterrows():
            # Add missing countries with 0 usage (both count and percentage)
            for country_code in missing_countries:
                # Add usage_count = 0
                new_rows.append(
                    {
                        "geo_id": country_code,
                        "geography": "country",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "country",
                        "level": 0,
                        "variable": "usage_count",
                        "cluster_name": "",
                        "value": 0.0,
                    }
                )
                # Add usage_pct = 0
                new_rows.append(
                    {
                        "geo_id": country_code,
                        "geography": "country",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "country",
                        "level": 0,
                        "variable": "usage_pct",
                        "cluster_name": "",
                        "value": 0.0,
                    }
                )

            # Add population data for all countries (that are not excluded)
            for _, pop_row in pop_country.iterrows():
                new_rows.append(
                    {
                        "geo_id": pop_row["country_code"],
                        "geography": "country",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "country",
                        "level": 0,
                        "variable": "working_age_pop",
                        "cluster_name": "",
                        "value": float(pop_row["working_age_pop"]),
                    }
                )

    # Process US states
    if "state_us" in population_data and not population_data["state_us"].empty:
        pop_state = population_data["state_us"]

        # Get existing states in our data
        existing_states = df_result[
            (df_result["geography"] == "state_us")
            & (df_result["variable"] == "usage_count")
        ]["geo_id"].unique()

        # Add missing states with 0 usage
        missing_states = set(pop_state["state_code"]) - set(existing_states)

        for _, combo in date_platform_combos.iterrows():
            # Add missing states with 0 usage (both count and percentage)
            for state_code in missing_states:
                # Add usage_count = 0
                new_rows.append(
                    {
                        "geo_id": state_code,
                        "geography": "state_us",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "state_us",
                        "level": 0,
                        "variable": "usage_count",
                        "cluster_name": "",
                        "value": 0.0,
                    }
                )
                # Add usage_pct = 0
                new_rows.append(
                    {
                        "geo_id": state_code,
                        "geography": "state_us",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "state_us",
                        "level": 0,
                        "variable": "usage_pct",
                        "cluster_name": "",
                        "value": 0.0,
                    }
                )

            # Add population data for all states
            for _, pop_row in pop_state.iterrows():
                new_rows.append(
                    {
                        "geo_id": pop_row["state_code"],
                        "geography": "state_us",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "state_us",
                        "level": 0,
                        "variable": "working_age_pop",
                        "cluster_name": "",
                        "value": float(pop_row["working_age_pop"]),
                    }
                )

    # Add all new rows to the dataframe
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        df_result = pd.concat([df_result, df_new], ignore_index=True)

    return df_result


def merge_gdp_data(df, gdp_data, population_data):
    """
    Merge GDP data and calculate GDP per working age capita.

    Since we have total GDP in actual dollars, we divide by population to get per capita.

    Args:
        df: Input dataframe in long format
        gdp_data: Dict with country and state_us GDP dataframes (total GDP in dollars)
        population_data: Dict with country and state_us population dataframes

    Returns:
        Dataframe with GDP per capita data added as rows
    """
    df_result = df.copy()
    new_rows = []

    # Get unique date/platform combinations
    date_platform_combos = df_result[
        ["date_start", "date_end", "platform_and_product"]
    ].drop_duplicates()

    # Process country GDP
    if "country" in gdp_data and "country" in population_data:
        gdp_country = gdp_data["country"]
        pop_country = population_data["country"]

        # Merge GDP with population to calculate per capita
        gdp_pop = gdp_country.merge(pop_country, on="iso_alpha_3", how="inner")

        # Calculate GDP per working age capita
        gdp_pop["gdp_per_working_age_capita"] = (
            gdp_pop["gdp_total"] / gdp_pop["working_age_pop"]
        )

        for _, combo in date_platform_combos.iterrows():
            for _, gdp_row in gdp_pop.iterrows():
                new_rows.append(
                    {
                        "geo_id": gdp_row["country_code"],  # Use 2-letter code
                        "geography": "country",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "country",
                        "level": 0,
                        "variable": "gdp_per_working_age_capita",
                        "cluster_name": "",
                        "value": float(gdp_row["gdp_per_working_age_capita"]),
                    }
                )

    # Process state GDP
    if "state_us" in gdp_data and "state_us" in population_data:
        gdp_state = gdp_data["state_us"]
        pop_state = population_data["state_us"]

        # Merge GDP with population
        # Column names from preprocess_gdp.py: state_code, gdp_total (in actual dollars)
        gdp_pop = gdp_state.merge(pop_state, on="state_code", how="inner")

        # Calculate GDP per working age capita
        gdp_pop["gdp_per_working_age_capita"] = (
            gdp_pop["gdp_total"] / gdp_pop["working_age_pop"]
        )

        for _, combo in date_platform_combos.iterrows():
            for _, gdp_row in gdp_pop.iterrows():
                new_rows.append(
                    {
                        "geo_id": gdp_row["state_code"],
                        "geography": "state_us",
                        "date_start": combo["date_start"],
                        "date_end": combo["date_end"],
                        "platform_and_product": combo["platform_and_product"],
                        "facet": "state_us",
                        "level": 0,
                        "variable": "gdp_per_working_age_capita",
                        "cluster_name": "",
                        "value": float(gdp_row["gdp_per_working_age_capita"]),
                    }
                )

    # Add all new rows to the dataframe
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        df_result = pd.concat([df_result, df_new], ignore_index=True)

    return df_result


def calculate_soc_distribution(
    df, df_onet, df_soc_structure, filtered_countries=None, filtered_states=None
):
    """
    Calculate SOC occupation distribution from O*NET task usage.

    This uses the following approach:
    1. Map tasks directly to SOC major groups (with minimal double counting)
    2. Combine "none" and "not_classified" tasks into a single "not_classified" SOC group
    3. Sum percentages by SOC group
    4. Normalize to 100% for each geography
    5. Calculate for countries, US states, and global that meet MIN_OBSERVATIONS threshold

    NOTE: For US states, only ~449 O*NET tasks have state-level data (those with sufficient
    observations), but these tasks still map to SOC groups the same way as for countries.

    Args:
        df: DataFrame with O*NET task percentages
        df_onet: O*NET task data with SOC codes
        df_soc_structure: SOC structure with major group names
        filtered_countries: List of countries that meet MIN_OBSERVATIONS (optional)
        filtered_states: List of states that meet MIN_OBSERVATIONS (optional)

    Returns:
        DataFrame with SOC distribution rows added
    """
    df_result = df.copy()
    soc_rows = []

    # Get all O*NET task percentage data (including not_classified and "none")
    df_task_pct_all = df_result[
        (df_result["facet"] == "onet_task") & (df_result["variable"] == "onet_task_pct")
    ].copy()

    if df_task_pct_all.empty:
        return df_result

    # Build masks for each geography type
    # Always include global
    global_mask = df_task_pct_all["geography"] == "global"

    # Apply filtering for countries
    if filtered_countries is not None:
        country_mask = (df_task_pct_all["geography"] == "country") & (
            df_task_pct_all["geo_id"].isin(filtered_countries)
        )
    else:
        # If no filter, keep all countries
        country_mask = df_task_pct_all["geography"] == "country"

    # Apply filtering for states
    if filtered_states is not None:
        state_mask = (df_task_pct_all["geography"] == "state_us") & (
            df_task_pct_all["geo_id"].isin(filtered_states)
        )
    else:
        # If no filter, keep all states
        state_mask = df_task_pct_all["geography"] == "state_us"

    # Combine masks to keep relevant geographies
    combined_mask = global_mask | country_mask | state_mask
    df_task_pct_all = df_task_pct_all[combined_mask].copy()

    if df_task_pct_all.empty:
        return df_result

    # Separate not_classified and none tasks from real O*NET tasks
    df_not_classified = df_task_pct_all[
        (df_task_pct_all["cluster_name"].str.contains("not_classified", na=False))
        | (df_task_pct_all["cluster_name"] == "none")
    ].copy()

    # Get real O*NET tasks (excluding not_classified and none)
    df_task_pct = df_task_pct_all[
        (~df_task_pct_all["cluster_name"].str.contains("not_classified", na=False))
        & (df_task_pct_all["cluster_name"] != "none")
    ].copy()

    # Normalize task names for matching
    df_task_pct["task_normalized"] = df_task_pct["cluster_name"].str.lower().str.strip()

    # Get unique task-SOC pairs from O*NET data
    # This keeps tasks that map to multiple SOC groups (different rows)
    df_task_soc = df_onet[["task_normalized", "soc_major_group"]].drop_duplicates()

    # Merge tasks with their SOC codes
    df_with_soc = df_task_pct.merge(df_task_soc, on="task_normalized", how="left")

    # Check for unmapped tasks and raise error if found (same as for countries)
    unmapped_tasks = df_with_soc[df_with_soc["soc_major_group"].isna()]
    if not unmapped_tasks.empty:
        unmapped_list = unmapped_tasks["cluster_name"].unique()[:10]  # Show first 10
        n_unmapped = len(unmapped_tasks["cluster_name"].unique())

        # Check which geographies have unmapped tasks
        unmapped_geos = unmapped_tasks["geography"].unique()

        raise ValueError(
            f"Found {n_unmapped} O*NET tasks that could not be mapped to SOC codes.\n"
            f"Geographies with unmapped tasks: {unmapped_geos.tolist()}\n"
            f"First 10 unmapped tasks:\n"
            + "\n".join(f"  - {task}" for task in unmapped_list)
            + f"\n\nThis likely means the O*NET data is out of sync with the Clio task data.\n"
            f"Please verify that preprocess_onet.py has been run with the correct O*NET version."
        )

    # Create SOC name mapping if SOC structure is available
    soc_names = {}
    if not df_soc_structure.empty:
        for _, row in df_soc_structure.iterrows():
            soc_code = row["soc_major_group"]
            title = row["SOC or O*NET-SOC 2019 Title"]
            # Clean up title (remove "Occupations" suffix)
            clean_title = title.replace(" Occupations", "").replace(" Occupation", "")
            soc_names[soc_code] = clean_title

    # Group by geography and process each group
    geo_groups = df_with_soc.groupby(
        ["geo_id", "geography", "date_start", "date_end", "platform_and_product"]
    )

    # Also group not_classified data by geography
    not_classified_groups = df_not_classified.groupby(
        ["geo_id", "geography", "date_start", "date_end", "platform_and_product"]
    )

    # Track statistics
    states_with_soc = set()
    countries_with_soc = set()

    # Process all geographies
    all_geos = set()
    for (geo_id, geography, date_start, date_end, platform), _ in geo_groups:
        all_geos.add((geo_id, geography, date_start, date_end, platform))
    for (geo_id, geography, date_start, date_end, platform), _ in not_classified_groups:
        all_geos.add((geo_id, geography, date_start, date_end, platform))

    for geo_id, geography, date_start, date_end, platform in all_geos:
        # Get mapped SOC data for this geography
        try:
            geo_data = geo_groups.get_group(
                (geo_id, geography, date_start, date_end, platform)
            )
            # Sum percentages by SOC major group
            # If a task maps to multiple SOC groups, its percentage is added to each
            soc_totals = geo_data.groupby("soc_major_group")["value"].sum()
        except KeyError:
            # No mapped tasks for this geography
            soc_totals = pd.Series(dtype=float)

        # Get not_classified/none data for this geography
        try:
            not_classified_data = not_classified_groups.get_group(
                (geo_id, geography, date_start, date_end, platform)
            )
            # Sum all not_classified and none percentages
            not_classified_total = not_classified_data["value"].sum()
        except KeyError:
            # No not_classified/none for this geography
            not_classified_total = 0

        # Combine and normalize to 100%
        total_pct = soc_totals.sum() + not_classified_total

        if total_pct > 0:
            # Normalize mapped SOC groups
            if len(soc_totals) > 0:
                soc_normalized = (soc_totals / total_pct) * 100
            else:
                soc_normalized = pd.Series(dtype=float)

            # Calculate normalized not_classified percentage
            not_classified_normalized = (not_classified_total / total_pct) * 100

            # Track geographies that have SOC data
            if geography == "state_us":
                states_with_soc.add(geo_id)
            elif geography == "country":
                countries_with_soc.add(geo_id)

            # Create rows for each SOC group
            for soc_group, pct_value in soc_normalized.items():
                # Get SOC name if available, otherwise use code
                soc_name = soc_names.get(soc_group, f"SOC {soc_group}")

                soc_row = {
                    "geo_id": geo_id,
                    "geography": geography,
                    "date_start": date_start,
                    "date_end": date_end,
                    "platform_and_product": platform,
                    "facet": "soc_occupation",
                    "level": 0,
                    "variable": "soc_pct",
                    "cluster_name": soc_name,
                    "value": pct_value,
                }
                soc_rows.append(soc_row)

            # Add not_classified SOC row if there's any not_classified/none percentage
            if not_classified_normalized > 0:
                soc_row = {
                    "geo_id": geo_id,
                    "geography": geography,
                    "date_start": date_start,
                    "date_end": date_end,
                    "platform_and_product": platform,
                    "facet": "soc_occupation",
                    "level": 0,
                    "variable": "soc_pct",
                    "cluster_name": "not_classified",
                    "value": not_classified_normalized,
                }
                soc_rows.append(soc_row)

    # Print summary
    if countries_with_soc:
        print(
            f"Calculated SOC distributions for {len(countries_with_soc)} countries + global"
        )
    if states_with_soc:
        print(f"Calculated SOC distributions for {len(states_with_soc)} US states")

    # Add all SOC rows to result
    if soc_rows:
        df_soc = pd.DataFrame(soc_rows)
        df_result = pd.concat([df_result, df_soc], ignore_index=True)

    return df_result

## Metric Calculation Functions

In [ ]:
def calculate_per_capita_metrics(df):
    """
    Calculate per capita metrics by joining usage and population data.

    Since data is in long format, this function:
    1. Extracts usage count rows
    2. Extracts population rows
    3. Joins them and calculates per capita
    4. Adds results as new rows

    Args:
        df: Dataframe in long format with usage and population as rows

    Returns:
        Dataframe with per capita metrics added as new rows
    """
    df_result = df.copy()

    # Define which metrics should have per capita calculations
    count_metrics = ["usage_count"]

    # Get population data
    df_pop = df_result[df_result["variable"] == "working_age_pop"][
        [
            "geo_id",
            "geography",
            "date_start",
            "date_end",
            "platform_and_product",
            "value",
        ]
    ].rename(columns={"value": "population"})

    # Calculate per capita for each count metric
    per_capita_rows = []

    for metric in count_metrics:
        # Get the count data for this metric
        df_metric = df_result[df_result["variable"] == metric].copy()

        # Join with population data
        df_joined = df_metric.merge(
            df_pop,
            on=[
                "geo_id",
                "geography",
                "date_start",
                "date_end",
                "platform_and_product",
            ],
            how="left",
        )

        # Calculate per capita where population exists and is > 0
        df_joined = df_joined[
            df_joined["population"].notna() & (df_joined["population"] > 0)
        ]

        if not df_joined.empty:
            # Create per capita rows
            for _, row in df_joined.iterrows():
                per_capita_row = {
                    "geo_id": row["geo_id"],
                    "geography": row["geography"],
                    "date_start": row["date_start"],
                    "date_end": row["date_end"],
                    "platform_and_product": row["platform_and_product"],
                    "facet": row["facet"],
                    "level": row["level"],
                    "variable": metric.replace("_count", "_per_capita"),
                    "cluster_name": row["cluster_name"],
                    "value": row["value"] / row["population"],
                }
                per_capita_rows.append(per_capita_row)

    # Add all per capita rows to the result
    if per_capita_rows:
        df_per_capita = pd.DataFrame(per_capita_rows)
        df_result = pd.concat([df_result, df_per_capita], ignore_index=True)

    return df_result


def calculate_usage_per_capita_index(df, filtered_countries=None, filtered_states=None):
    """
    Calculate usage concentration index: (% of usage) / (% of population).

    This shows whether a geography has more or less usage than expected based on its population.
    - Index = 1.0: Usage proportional to population
    - Index > 1.0: Over-representation (more usage than expected)
    - Index < 1.0: Under-representation (less usage than expected)
    - Index = 0.0: No usage at all

    The function calculates the index for all countries/states that have usage data.
    Excluded countries don't have usage data, so they're automatically excluded.
    Countries with zero usage get index=0 naturally from the calculation.

    Args:
        df: Dataframe with usage and population data
        filtered_countries: List of countries that meet MIN_OBSERVATIONS threshold (used for baseline calculation)
        filtered_states: List of states that meet MIN_OBSERVATIONS threshold (used for baseline calculation)

    Returns:
        Dataframe with usage concentration index added as new rows
    """
    df_result = df.copy()

    index_rows = []

    # Process countries
    # Get all countries with usage data (excluded countries won't be here)
    df_usage_country = df_result[
        (df_result["geography"] == "country") & (df_result["variable"] == "usage_count")
    ].copy()

    # Get population data for the same countries
    df_pop_country = df_result[
        (df_result["geography"] == "country")
        & (df_result["variable"] == "working_age_pop")
    ].copy()

    if not df_usage_country.empty and not df_pop_country.empty:
        # For baseline calculation, use filtered countries if provided, otherwise use all
        if filtered_countries is not None:
            # Calculate totals using only filtered countries for the baseline
            usage_for_baseline = df_usage_country[
                df_usage_country["geo_id"].isin(filtered_countries)
            ]
            pop_for_baseline = df_pop_country[
                df_pop_country["geo_id"].isin(filtered_countries)
            ]
            total_usage = usage_for_baseline["value"].sum()
            total_pop = pop_for_baseline["value"].sum()
        else:
            # Use all countries for baseline
            total_usage = df_usage_country["value"].sum()
            total_pop = df_pop_country["value"].sum()

        if total_usage > 0 and total_pop > 0:
            # Calculate index for all countries (not just filtered)
            for _, usage_row in df_usage_country.iterrows():
                # Find corresponding population
                pop_value = df_pop_country[
                    df_pop_country["geo_id"] == usage_row["geo_id"]
                ]["value"].values

                if len(pop_value) > 0 and pop_value[0] > 0:
                    # Calculate shares
                    usage_share = (
                        usage_row["value"] / total_usage
                        if usage_row["value"] > 0
                        else 0
                    )
                    pop_share = pop_value[0] / total_pop

                    # Calculate index (will be 0 if usage is 0)
                    index_value = usage_share / pop_share if pop_share > 0 else 0

                    index_row = {
                        "geo_id": usage_row["geo_id"],
                        "geography": usage_row["geography"],
                        "date_start": usage_row["date_start"],
                        "date_end": usage_row["date_end"],
                        "platform_and_product": usage_row["platform_and_product"],
                        "facet": usage_row["facet"],
                        "level": usage_row["level"],
                        "variable": "usage_per_capita_index",
                        "cluster_name": usage_row["cluster_name"],
                        "value": index_value,
                    }
                    index_rows.append(index_row)

    # Process states
    # Get all states with usage data
    df_usage_state = df_result[
        (df_result["geography"] == "state_us")
        & (df_result["variable"] == "usage_count")
    ].copy()

    # Get population data for the same states
    df_pop_state = df_result[
        (df_result["geography"] == "state_us")
        & (df_result["variable"] == "working_age_pop")
    ].copy()

    if not df_usage_state.empty and not df_pop_state.empty:
        # For baseline calculation, use filtered states if provided, otherwise use all
        if filtered_states is not None:
            # Calculate totals using only filtered states for the baseline
            usage_for_baseline = df_usage_state[
                df_usage_state["geo_id"].isin(filtered_states)
            ]
            pop_for_baseline = df_pop_state[
                df_pop_state["geo_id"].isin(filtered_states)
            ]
            total_usage = usage_for_baseline["value"].sum()
            total_pop = pop_for_baseline["value"].sum()
        else:
            # Use all states for baseline
            total_usage = df_usage_state["value"].sum()
            total_pop = df_pop_state["value"].sum()

        if total_usage > 0 and total_pop > 0:
            # Calculate index for all states (not just filtered)
            for _, usage_row in df_usage_state.iterrows():
                # Find corresponding population
                pop_value = df_pop_state[df_pop_state["geo_id"] == usage_row["geo_id"]][
                    "value"
                ].values

                if len(pop_value) > 0 and pop_value[0] > 0:
                    # Calculate shares
                    usage_share = (
                        usage_row["value"] / total_usage
                        if usage_row["value"] > 0
                        else 0
                    )
                    pop_share = pop_value[0] / total_pop

                    # Calculate index (will be 0 if usage is 0)
                    index_value = usage_share / pop_share if pop_share > 0 else 0

                    index_row = {
                        "geo_id": usage_row["geo_id"],
                        "geography": usage_row["geography"],
                        "date_start": usage_row["date_start"],
                        "date_end": usage_row["date_end"],
                        "platform_and_product": usage_row["platform_and_product"],
                        "facet": usage_row["facet"],
                        "level": usage_row["level"],
                        "variable": "usage_per_capita_index",
                        "cluster_name": usage_row["cluster_name"],
                        "value": index_value,
                    }
                    index_rows.append(index_row)

    # Add all index rows to result
    if index_rows:
        df_index = pd.DataFrame(index_rows)
        df_result = pd.concat([df_result, df_index], ignore_index=True)

    return df_result


def calculate_category_percentage_index(
    df, filtered_countries=None, filtered_states=None
):
    """
    Calculate category percentage index for facet specialization.

    For countries: Compare to global percentage for that cluster
    For US states: Compare to US country percentage for that cluster

    Only calculates for countries/states that meet MIN_OBSERVATIONS.
    Excludes "not_classified" and "none" categories as these are catch-alls.

    Args:
        df: Dataframe with percentage metrics as rows
        filtered_countries: List of countries that meet MIN_OBSERVATIONS threshold
        filtered_states: List of states that meet MIN_OBSERVATIONS threshold

    Returns:
        Dataframe with category percentage index added as new rows (only for filtered geographies)
    """
    df_result = df.copy()

    # Process percentage metrics for content facets
    pct_vars = ["onet_task_pct", "collaboration_pct", "request_pct"]

    index_rows = []

    for pct_var in pct_vars:
        # Get the base facet name
        facet_name = pct_var.replace("_pct", "")

        # Get percentage data for this variable
        df_pct = df_result[
            (df_result["variable"] == pct_var) & (df_result["facet"] == facet_name)
        ].copy()

        # Exclude not_classified and none categories from index calculation
        # These are catch-all/no-pattern categories that don't provide meaningful comparisons
        df_pct = df_pct[~df_pct["cluster_name"].isin(["not_classified", "none"])]

        if not df_pct.empty and "cluster_name" in df_pct.columns:
            # Check if this facet has levels (like request)
            has_levels = df_pct["level"].notna().any() and (df_pct["level"] != 0).any()

            if has_levels:
                # Process each level separately
                levels = df_pct["level"].dropna().unique()

                for level in levels:
                    df_level = df_pct[df_pct["level"] == level].copy()

                    # Get global baselines for this level
                    global_baselines = (
                        df_level[
                            (df_level["geography"] == "global")
                            & (df_level["geo_id"] == "GLOBAL")
                        ]
                        .set_index("cluster_name")["value"]
                        .to_dict()
                    )

                    # Get US baselines for this level
                    us_baselines = (
                        df_level[
                            (df_level["geography"] == "country")
                            & (df_level["geo_id"] == "US")
                        ]
                        .set_index("cluster_name")["value"]
                        .to_dict()
                    )

                    # Process countries for this level
                    if filtered_countries is not None and global_baselines:
                        df_countries = df_level[
                            (df_level["geography"] == "country")
                            & (df_level["geo_id"].isin(filtered_countries))
                        ].copy()

                        for _, row in df_countries.iterrows():
                            baseline = global_baselines.get(row["cluster_name"])

                            if baseline and baseline > 0:
                                index_row = {
                                    "geo_id": row["geo_id"],
                                    "geography": row["geography"],
                                    "date_start": row["date_start"],
                                    "date_end": row["date_end"],
                                    "platform_and_product": row["platform_and_product"],
                                    "facet": row["facet"],
                                    "level": row["level"],
                                    "variable": f"{facet_name}_pct_index",
                                    "cluster_name": row["cluster_name"],
                                    "value": row["value"] / baseline,
                                }
                                index_rows.append(index_row)

                    # Process states for this level
                    if filtered_states is not None and us_baselines:
                        df_states = df_level[
                            (df_level["geography"] == "state_us")
                            & (df_level["geo_id"].isin(filtered_states))
                        ].copy()

                        for _, row in df_states.iterrows():
                            baseline = us_baselines.get(row["cluster_name"])

                            if baseline and baseline > 0:
                                index_row = {
                                    "geo_id": row["geo_id"],
                                    "geography": row["geography"],
                                    "date_start": row["date_start"],
                                    "date_end": row["date_end"],
                                    "platform_and_product": row["platform_and_product"],
                                    "facet": row["facet"],
                                    "level": row["level"],
                                    "variable": f"{facet_name}_pct_index",
                                    "cluster_name": row["cluster_name"],
                                    "value": row["value"] / baseline,
                                }
                                index_rows.append(index_row)
            else:
                # No levels (onet_task, collaboration)
                # Get global baselines
                global_baselines = (
                    df_pct[
                        (df_pct["geography"] == "global")
                        & (df_pct["geo_id"] == "GLOBAL")
                    ]
                    .set_index("cluster_name")["value"]
                    .to_dict()
                )

                # Get US baselines
                us_baselines = (
                    df_pct[
                        (df_pct["geography"] == "country") & (df_pct["geo_id"] == "US")
                    ]
                    .set_index("cluster_name")["value"]
                    .to_dict()
                )

                # Process countries
                if filtered_countries is not None and global_baselines:
                    df_countries = df_pct[
                        (df_pct["geography"] == "country")
                        & (df_pct["geo_id"].isin(filtered_countries))
                    ].copy()

                    for _, row in df_countries.iterrows():
                        baseline = global_baselines.get(row["cluster_name"])

                        if baseline and baseline > 0:
                            index_row = {
                                "geo_id": row["geo_id"],
                                "geography": row["geography"],
                                "date_start": row["date_start"],
                                "date_end": row["date_end"],
                                "platform_and_product": row["platform_and_product"],
                                "facet": row["facet"],
                                "level": row["level"],
                                "variable": f"{facet_name}_pct_index",
                                "cluster_name": row["cluster_name"],
                                "value": row["value"] / baseline,
                            }
                            index_rows.append(index_row)

                # Process states
                if filtered_states is not None and us_baselines:
                    df_states = df_pct[
                        (df_pct["geography"] == "state_us")
                        & (df_pct["geo_id"].isin(filtered_states))
                    ].copy()

                    for _, row in df_states.iterrows():
                        baseline = us_baselines.get(row["cluster_name"])

                        if baseline and baseline > 0:
                            index_row = {
                                "geo_id": row["geo_id"],
                                "geography": row["geography"],
                                "date_start": row["date_start"],
                                "date_end": row["date_end"],
                                "platform_and_product": row["platform_and_product"],
                                "facet": row["facet"],
                                "level": row["level"],
                                "variable": f"{facet_name}_pct_index",
                                "cluster_name": row["cluster_name"],
                                "value": row["value"] / baseline,
                            }
                            index_rows.append(index_row)

    # Add all index rows to result
    if index_rows:
        df_index = pd.DataFrame(index_rows)
        df_result = pd.concat([df_result, df_index], ignore_index=True)

    return df_result

In [ ]:
def calculate_usage_tiers(df, n_tiers=4, filtered_countries=None, filtered_states=None):
    """
    Calculate usage tiers based on indexed per capita usage.
    - Tier 0: Zero adoption (index = 0)
    - Tiers 1-4: Quartiles based on thresholds from filtered countries/states

    Quartile thresholds are calculated using only countries/states with ≥MIN_OBSERVATIONS,
    but applied to all countries/states to ensure complete visualization.

    Note: Tier assignments for countries/states with <MIN_OBSERVATIONS should be
    interpreted with caution due to sample size limitations.

    Args:
        df: Input dataframe
        n_tiers: Number of quartiles to create for non-zero usage (default 4)
        filtered_countries: List of countries that meet MIN_OBSERVATIONS threshold
        filtered_states: List of states that meet MIN_OBSERVATIONS threshold

    Returns:
        Dataframe with usage tier rows added
    """
    df_result = df.copy()

    # Calculate tiers for indexed per capita metrics
    if "variable" in df_result.columns and "value" in df_result.columns:
        index_vars = ["usage_per_capita_index"]

        quartile_labels = [
            "Emerging (bottom 25%)",
            "Lower middle (25-50%)",
            "Upper middle (50-75%)",
            "Leading (top 25%)",
        ]

        tier_rows = []

        for var in index_vars:
            # Process countries
            # Get all countries with the index variable
            all_country_data = df_result[
                (df_result["variable"] == var) & (df_result["geography"] == "country")
            ].copy()

            if not all_country_data.empty:
                # Separate zero and non-zero usage
                zero_usage = all_country_data[all_country_data["value"] == 0].copy()
                nonzero_usage = all_country_data[all_country_data["value"] > 0].copy()

                # Calculate quartile thresholds using ONLY filtered countries
                if filtered_countries is not None and not nonzero_usage.empty:
                    # Get only filtered countries for quartile calculation
                    filtered_for_quartiles = nonzero_usage[
                        nonzero_usage["geo_id"].isin(filtered_countries)
                    ].copy()

                    if not filtered_for_quartiles.empty:
                        # Calculate quartile thresholds from filtered countries
                        quartiles = (
                            filtered_for_quartiles["value"]
                            .quantile([0.25, 0.5, 0.75])
                            .values
                        )

                        # Apply thresholds to all non-zero countries
                        for _, row in nonzero_usage.iterrows():
                            value = row["value"]

                            # Assign tier based on thresholds
                            if value <= quartiles[0]:
                                tier_label = quartile_labels[0]  # Bottom 25%
                                tier_value = 1
                            elif value <= quartiles[1]:
                                tier_label = quartile_labels[1]  # 25-50%
                                tier_value = 2
                            elif value <= quartiles[2]:
                                tier_label = quartile_labels[2]  # 50-75%
                                tier_value = 3
                            else:
                                tier_label = quartile_labels[3]  # Top 25%
                                tier_value = 4

                            tier_row = {
                                "geo_id": row["geo_id"],
                                "geography": row["geography"],
                                "date_start": row["date_start"],
                                "date_end": row["date_end"],
                                "platform_and_product": row["platform_and_product"],
                                "facet": row["facet"],
                                "level": row["level"],
                                "variable": "usage_tier",
                                "cluster_name": tier_label,
                                "value": tier_value,
                            }
                            tier_rows.append(tier_row)

                # Add tier 0 for all zero usage countries
                for _, row in zero_usage.iterrows():
                    tier_row = {
                        "geo_id": row["geo_id"],
                        "geography": row["geography"],
                        "date_start": row["date_start"],
                        "date_end": row["date_end"],
                        "platform_and_product": row["platform_and_product"],
                        "facet": row["facet"],
                        "level": row["level"],
                        "variable": "usage_tier",
                        "cluster_name": "Minimal",
                        "value": 0,
                    }
                    tier_rows.append(tier_row)

            # Process states
            # Get all states with the index variable
            all_state_data = df_result[
                (df_result["variable"] == var) & (df_result["geography"] == "state_us")
            ].copy()

            if not all_state_data.empty:
                # Separate zero and non-zero usage
                zero_usage = all_state_data[all_state_data["value"] == 0].copy()
                nonzero_usage = all_state_data[all_state_data["value"] > 0].copy()

                # Calculate quartile thresholds using ONLY filtered states
                if filtered_states is not None and not nonzero_usage.empty:
                    # Get only filtered states for quartile calculation
                    filtered_for_quartiles = nonzero_usage[
                        nonzero_usage["geo_id"].isin(filtered_states)
                    ].copy()

                    if not filtered_for_quartiles.empty:
                        # Calculate quartile thresholds from filtered states
                        quartiles = (
                            filtered_for_quartiles["value"]
                            .quantile([0.25, 0.5, 0.75])
                            .values
                        )

                        # Apply thresholds to all non-zero states
                        for _, row in nonzero_usage.iterrows():
                            value = row["value"]

                            # Assign tier based on thresholds
                            if value <= quartiles[0]:
                                tier_label = quartile_labels[0]  # Bottom 25%
                                tier_value = 1
                            elif value <= quartiles[1]:
                                tier_label = quartile_labels[1]  # 25-50%
                                tier_value = 2
                            elif value <= quartiles[2]:
                                tier_label = quartile_labels[2]  # 50-75%
                                tier_value = 3
                            else:
                                tier_label = quartile_labels[3]  # Top 25%
                                tier_value = 4

                            tier_row = {
                                "geo_id": row["geo_id"],
                                "geography": row["geography"],
                                "date_start": row["date_start"],
                                "date_end": row["date_end"],
                                "platform_and_product": row["platform_and_product"],
                                "facet": row["facet"],
                                "level": row["level"],
                                "variable": "usage_tier",
                                "cluster_name": tier_label,
                                "value": tier_value,
                            }
                            tier_rows.append(tier_row)

                # Add tier 0 for all zero usage states
                for _, row in zero_usage.iterrows():
                    tier_row = {
                        "geo_id": row["geo_id"],
                        "geography": row["geography"],
                        "date_start": row["date_start"],
                        "date_end": row["date_end"],
                        "platform_and_product": row["platform_and_product"],
                        "facet": row["facet"],
                        "level": row["level"],
                        "variable": "usage_tier",
                        "cluster_name": "Minimal",
                        "value": 0,
                    }
                    tier_rows.append(tier_row)

        if tier_rows:
            df_result = pd.concat(
                [df_result, pd.DataFrame(tier_rows)], ignore_index=True
            )

    return df_result

In [ ]:
def calculate_automation_augmentation_metrics(
    df, filtered_countries=None, filtered_states=None
):
    """
    Calculate automation vs augmentation percentages for collaboration patterns.

    This function:
    1. Categorizes collaboration patterns as automation or augmentation
    2. Calculates percentages excluding 'none' and 'not_classified'
    3. Only calculates for filtered geographies at country/state level

    Categorization:
    - Automation: directive, feedback loop (AI-centric patterns)
    - Augmentation: validation, task iteration, learning (human-centric patterns)
    - Excluded: none (no collaboration), not_classified (unknown)

    Args:
        df: Dataframe with collaboration data
        filtered_countries: List of countries that meet MIN_OBSERVATIONS
        filtered_states: List of states that meet MIN_OBSERVATIONS

    Returns:
        Dataframe with automation/augmentation percentage rows added
    """
    if "facet" not in df.columns or "cluster_name" not in df.columns:
        return df

    df_result = df.copy()

    # Get collaboration data
    collab_data = df_result[
        (df_result["facet"] == "collaboration")
        & (df_result["variable"] == "collaboration_count")
    ].copy()

    if collab_data.empty:
        return df_result

    # Define pattern categorization
    def categorize_pattern(pattern_name):
        if pd.isna(pattern_name):
            return None

        pattern_clean = pattern_name.lower().replace("_", " ").replace("-", " ")

        # Augmentation patterns (human-centric)
        if "validation" in pattern_clean:
            return "augmentation"
        elif "task iteration" in pattern_clean or "task_iteration" in pattern_clean:
            return "augmentation"
        elif "learning" in pattern_clean:
            return "augmentation"
        # Automation patterns (AI-centric)
        elif "directive" in pattern_clean:
            return "automation"
        elif "feedback loop" in pattern_clean or "feedback_loop" in pattern_clean:
            return "automation"
        # Excluded patterns - return None to exclude from calculations
        elif "none" in pattern_clean or "not_classified" in pattern_clean:
            return None
        else:
            return None  # Unknown patterns also excluded

    # Add category column
    collab_data["category"] = collab_data["cluster_name"].apply(categorize_pattern)

    # Filter to only patterns that have a category (excludes none, not_classified, etc.)
    collab_categorized = collab_data[collab_data["category"].notna()].copy()

    if collab_categorized.empty:
        return df_result

    # Process by geography
    new_rows = []

    # Group by geography and geo_id
    for (geography, geo_id), geo_data in collab_categorized.groupby(
        ["geography", "geo_id"]
    ):
        # Apply filtering based on geography level
        if geography == "country" and filtered_countries is not None:
            if geo_id not in filtered_countries:
                continue  # Skip countries that don't meet threshold
        elif geography == "state_us" and filtered_states is not None:
            if geo_id not in filtered_states:
                continue  # Skip states that don't meet threshold
        # global is always included (no filtering)

        # Calculate totals by category
        automation_total = geo_data[geo_data["category"] == "automation"]["value"].sum()
        augmentation_total = geo_data[geo_data["category"] == "augmentation"][
            "value"
        ].sum()

        # Total of categorized patterns (excluding none and not_classified)
        total_categorized = automation_total + augmentation_total

        if total_categorized > 0:
            # Get a sample row for metadata
            sample_row = geo_data.iloc[0]

            # Create automation percentage row
            automation_row = {
                "geo_id": geo_id,
                "geography": geography,
                "date_start": sample_row["date_start"],
                "date_end": sample_row["date_end"],
                "platform_and_product": sample_row["platform_and_product"],
                "facet": "collaboration_automation_augmentation",
                "level": 0,
                "variable": "automation_pct",
                "cluster_name": "automation",
                "value": (automation_total / total_categorized) * 100,
            }
            new_rows.append(automation_row)

            # Create augmentation percentage row
            augmentation_row = {
                "geo_id": geo_id,
                "geography": geography,
                "date_start": sample_row["date_start"],
                "date_end": sample_row["date_end"],
                "platform_and_product": sample_row["platform_and_product"],
                "facet": "collaboration_automation_augmentation",
                "level": 0,
                "variable": "augmentation_pct",
                "cluster_name": "augmentation",
                "value": (augmentation_total / total_categorized) * 100,
            }
            new_rows.append(augmentation_row)

    # Add all new rows to result
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        df_result = pd.concat([df_result, df_new], ignore_index=True)

    return df_result

In [ ]:
def add_iso3_and_names(df):
    """
    Replace ISO-2 codes with ISO-3 codes and add geographic names.

    This function:
    1. Replaces geo_id from ISO-2 to ISO-3 for countries
    2. Adds geo_name column with human-readable names for all geographies
    3. Preserves special geo_ids (like 'not_classified') that aren't in ISO mapping

    Args:
        df: Enriched dataframe with geo_id (ISO-2 for countries, state codes for US states)

    Returns:
        Dataframe with ISO-3 codes in geo_id and geo_name column added
    """
    df_result = df.copy()

    # Initialize geo_name column
    df_result["geo_name"] = ""

    # Load ISO mapping data for countries
    iso_path = Path(DATA_INTERMEDIATE_DIR) / "iso_country_codes.csv"
    if iso_path.exists():
        df_iso = pd.read_csv(iso_path, keep_default_na=False, na_values=[""])

        # Create ISO-2 to ISO-3 mapping
        iso2_to_iso3 = dict(zip(df_iso["iso_alpha_2"], df_iso["iso_alpha_3"]))

        # Create ISO-2 to country name mapping
        iso2_to_name = dict(zip(df_iso["iso_alpha_2"], df_iso["country_name"]))

        # For all rows where geography is 'country', add country names and convert codes
        # This includes content facets that are broken down by country
        country_mask = df_result["geography"] == "country"

        # First, identify which geo_ids don't have ISO mappings
        country_geo_ids = df_result.loc[country_mask, "geo_id"].unique()
        unmapped_geo_ids = [
            g for g in country_geo_ids if g not in iso2_to_iso3 and pd.notna(g)
        ]

        if unmapped_geo_ids:
            print(
                f"\nWarning: The following geo_ids are not in ISO-2 mapping and will be kept as-is:"
            )
            for geo_id in unmapped_geo_ids:
                # Count rows and usage for this geo_id
                geo_mask = (df_result["geography"] == "country") & (
                    df_result["geo_id"] == geo_id
                )
                row_count = geo_mask.sum()
                usage_mask = geo_mask & (df_result["variable"] == "usage_count")
                usage_sum = (
                    df_result.loc[usage_mask, "value"].sum() if usage_mask.any() else 0
                )
                print(f"  - '{geo_id}': {row_count} rows, {usage_sum:,.0f} usage count")

        # Check for geo_ids without country names
        unmapped_names = [g for g in unmapped_geo_ids if g not in iso2_to_name]
        if unmapped_names:
            print(
                f"\nWarning: The following geo_ids don't have country names and will use geo_id as name:"
            )
            for geo_id in unmapped_names:
                print(f"  - '{geo_id}'")

        # Apply country names BEFORE converting ISO-2 to ISO-3
        # The iso2_to_name dictionary uses ISO-2 codes as keys
        df_result.loc[country_mask, "geo_name"] = (
            df_result.loc[country_mask, "geo_id"]
            .map(iso2_to_name)
            .fillna(df_result.loc[country_mask, "geo_id"])
        )

        # Convert ISO-2 to ISO-3 codes
        df_result.loc[country_mask, "geo_id"] = (
            df_result.loc[country_mask, "geo_id"]
            .map(iso2_to_iso3)
            .fillna(df_result.loc[country_mask, "geo_id"])
        )
    else:
        print(f"Warning: ISO mapping file not found at {iso_path}")

    # Load state names from census data
    state_codes_path = Path(DATA_INPUT_DIR) / "census_state_codes.txt"
    if state_codes_path.exists():
        df_state_codes = pd.read_csv(state_codes_path, sep="|")
        # Create state code to name mapping (STUSAB is the 2-letter code, STATE_NAME is the full name)
        state_code_to_name = dict(
            zip(df_state_codes["STUSAB"], df_state_codes["STATE_NAME"])
        )

        # For all rows where geography is 'state_us', add state names
        state_mask = df_result["geography"] == "state_us"
        df_result.loc[state_mask, "geo_name"] = df_result.loc[state_mask, "geo_id"].map(
            state_code_to_name
        )
    else:
        print(f"Warning: State census file not found at {state_codes_path}")

    # For global entries
    global_mask = df_result["geography"] == "global"
    df_result.loc[global_mask, "geo_name"] = "global"

    # Fill any missing geo_names with geo_id as fallback
    df_result.loc[df_result["geo_name"] == "", "geo_name"] = df_result.loc[
        df_result["geo_name"] == "", "geo_id"
    ]
    df_result["geo_name"] = df_result["geo_name"].fillna(df_result["geo_id"])

    return df_result

## Main Processing Function

In [ ]:
def enrich_clio_data(input_path, output_path, external_data=None):
    """
    Enrich processed Clio data with external sources.

    Args:
        input_path: Path to processed Clio data
        output_path: Path for enriched CSV output
        external_data: Pre-loaded external data (optional)

    Returns:
        Path to enriched data file
    """
    # Load processed Clio data - use keep_default_na=False to preserve "NA" (Namibia)
    df = pd.read_csv(input_path, keep_default_na=False, na_values=[""])

    # Load external data if not provided
    if external_data is None:
        external_data = load_external_data()

    # Get filtered geographies (but keep all data in the dataframe)
    filtered_countries, filtered_states = get_filtered_geographies(df)

    # Merge with population data
    df = merge_population_data(df, external_data["population"])

    # Merge with GDP data (pass population data for per capita calculation)
    df = merge_gdp_data(df, external_data["gdp"], external_data["population"])

    # Calculate SOC occupation distribution from O*NET tasks
    # Only for geographies that meet MIN_OBSERVATIONS threshold
    df = calculate_soc_distribution(
        df,
        external_data["task_statements"],
        external_data["soc_structure"],
        filtered_countries=filtered_countries,
        filtered_states=filtered_states,
    )

    # Calculate per capita metrics
    df = calculate_per_capita_metrics(df)

    # Calculate usage index - pass filtered countries/states to only use them for baseline
    df = calculate_usage_per_capita_index(
        df, filtered_countries=filtered_countries, filtered_states=filtered_states
    )

    # Calculate category percentage index - pass filtered countries/states
    df = calculate_category_percentage_index(
        df, filtered_countries=filtered_countries, filtered_states=filtered_states
    )

    # Calculate usage tiers - pass filtered countries/states to only use them
    df = calculate_usage_tiers(
        df, filtered_countries=filtered_countries, filtered_states=filtered_states
    )

    # Add collaboration categorization
    df = calculate_automation_augmentation_metrics(df)

    # Add ISO-3 codes and geographic names
    df = add_iso3_and_names(df)

    # Sort for consistent output ordering
    df = df.sort_values(
        ["geography", "geo_id", "facet", "level", "cluster_name", "variable"]
    )

    # Save enriched data as CSV
    df.to_csv(output_path, index=False)

    return str(output_path)

## Merge External Data

In [ ]:
input_path = "../data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv"
output_path = "../data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv"

enrich_clio_data(input_path, output_path)
print(f"\n✅ Enrichment complete! Output: {output_path}")